# Pipeline PQRS - Superintendencia Financiera de Colombia

**Engagement:** Accenbank Group · Johanna (Accenture Banking Strategy & Consulting) · Sergio (Strategic Growth Manager)  
**Objetivo:** Transformar 1,044,109 registros trimestrales de PQRS en un dataset limpio, auditado y con trazabilidad completa, listo para sustentar decisiones frente a la Junta Directiva.

---

### Auditoria de fuente - hallazgo de integridad de metadatos

La documentacion interna de Accenture (PPTX del engagement) registra la variable `TIEMPO_PROMEDIO_EN_SISTEMA_(H)` como disponible en el dataset. **Esta columna no esta presente en el CSV entregado.** Se documenta aqui como hallazgo de auditoria de fuente antes de iniciar cualquier transformacion; demuestra criterio profesional y evita que analisis posteriores asuman disponibilidad de esa variable.

---

### Estructura del documento

| Seccion | Contenido |
|---|---|
| 0 | Setup y carga de datos |
| 1 | Diagnostico - Data Profiling |
| 2 | Limpieza con flags de trazabilidad |
| 3 | Variables derivadas |
| 4 | Exportacion del dataset limpio |
| 5 | Validacion post-limpieza |
| 6 | Hallazgos de negocio - entregable mandatorio |

---
*Fecha de ejecucion: ver output de celda 0.1*

## Seccion 0 - Setup y Carga de Datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import datetime

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')

RAW_PATH = os.path.join('data', 'raw', 'Challenge_DataAI_Candidatos.csv')
PROCESSED_PATH = os.path.join('data', 'processed', 'quejas_sfc_clean.parquet')

COLS_CLAVE = ['TIPO_ENTIDAD', 'CODIGO_ENTIDAD', 'FECHA_CORTE', 'PRODUCTO', 'MOTIVO']

print(f"Ejecucion: {datetime.date.today()}")

### 0.1 Carga del dataset

**Decisiones de ingesta:**

- **Separador `;`** - formato estandar de los exports de la SFC.
- **Encoding `latin1`** - el archivo viene en ISO-8859-1; pandas falla con `utf-8`/`utf-8-sig` porque encuentra bytes validos en Latin-1 pero invalidos para UTF-8. Se documenta como hallazgo de auditoria de fuente.
- **`dtype={'CODIGO_ENTIDAD': str}`** - el campo tiene valores mixtos (numericos y alfanumericos). Forzar string en ingesta preserva los valores originales para auditoria y evita que pandas descarte caracteres no numericos por coercion silenciosa.

**Validacion de volumen:** el archivo tiene 1,044,109 registros de negocio. Si el shape no coincide, el notebook se detiene con `AssertionError`.

In [ ]:
df = pd.read_csv(RAW_PATH, sep=';', encoding='latin1', dtype={'CODIGO_ENTIDAD': str})

# El CSV tiene saltos de linea Windows (\r\n) y columnas vacias al final.
# Se eliminan columnas sin nombre (Unnamed) y se limpian espacios en nombres de columna.
df.columns = df.columns.str.strip()
df = df.loc[:, ~df.columns.str.startswith('Unnamed')]

assert df.shape == (1_044_109, 18), f"Shape inesperado: {df.shape} - revisar archivo fuente"
print(f"Dataset cargado: {df.shape[0]:,} filas x {df.shape[1]} columnas")

display(df.head(5))
display(df.dtypes.rename('tipo_pandas').to_frame())
df.info(memory_usage='deep')

---

## Seccion 1 - Diagnostico (Data Profiling)

**Regla de esta seccion:** ningun dato es modificado. Solo se observa, cuantifica y documenta. Toda cifra aqui es evidencia que justifica las decisiones de limpieza de la Seccion 2.

**Bloques:**
1. Volumen, distribucion temporal y por entidad
2. Tipos de dato y deteccion de formatos mixtos
3. Cardinalidad y valores unicos en campos categoricos
4. Duplicados por clave de negocio
5. Los 7 problemas de calidad documentados
6. Validacion de identidades logicas PQRS
7. Distribucion de variables numericas clave
8. Preguntas de negocio exploratorias

### 1.1 Volumen, distribucion temporal y por entidad

Confirma el total de registros, cuantos periodos trimestrales distintos existen y cuales son las entidades con mayor volumen de reportes. Establece la escala del dataset antes de cualquier diagnostico de calidad.

In [ ]:
print(f"Total registros: {df.shape[0]:,} | Columnas: {df.shape[1]}")
print(f"Periodos distintos en FECHA_CORTE: {df['FECHA_CORTE'].nunique()}")
print(f"Entidades distintas en NOMBRE_ENTIDAD: {df['NOMBRE_ENTIDAD'].nunique()}")

print("\nTop 10 periodos por volumen de registros:")
display(df['FECHA_CORTE'].value_counts().head(10).rename('registros').to_frame())

print("\nTop 10 entidades por volumen de registros:")
display(df['NOMBRE_ENTIDAD'].value_counts().head(10).rename('registros').to_frame())

### Conclusion 1.1



### 1.2 Tipos de dato y deteccion de formatos mixtos

Compara el tipo inferido por pandas contra el tipo esperado de negocio. Responde explicitamente si `CODIGO_ENTIDAD` es siempre numerico y si `FECHA_CORTE` tiene un solo formato.

In [ ]:
tipos_esperados = {
    'TIPO_ENTIDAD': 'object', 'CODIGO_ENTIDAD': 'object', 'NOMBRE_ENTIDAD': 'object',
    'FECHA_CORTE': 'datetime64', 'UNIDAD_CAPTURA': 'object', 'NOMBRE_UNIDAD_CAPTURA': 'object',
    'CODIGO_PRODUCTO': 'int64', 'PRODUCTO': 'object', 'CODIGO_MOTIVO': 'int64',
    'MOTIVO': 'object', 'QUEJAS_PENDIENTES': 'int64', 'QUEJAS_RECIBIDAS': 'int64',
    'QUEJAS_FINALIZADAS': 'int64', 'QUEJAS_FINALIZADAS_N1': 'int64',
    'QUEJAS_FINALIZADAS_N2': 'int64', 'QUEJAS_EN_TRAMITE': 'int64',
    'QUEJAS_FAVOR_CONSUM_ACEP_ENT': 'int64', 'QUEJAS_FAVOR_CONSUM_NOACEP_ENT': 'int64'
}

tipo_report = pd.DataFrame({
    'tipo_pandas': df.dtypes.astype(str),
    'tipo_esperado': pd.Series(tipos_esperados, dtype='object')
})
tipo_report['coincide'] = [
    (str(esp) in str(real)) if (esp == esp and real == real) else False
    for real, esp in zip(tipo_report['tipo_pandas'], tipo_report['tipo_esperado'])
]
display(tipo_report)

# CODIGO_ENTIDAD: cuantos valores no son puramente numericos
mask_no_numerico = ~df['CODIGO_ENTIDAD'].str.isnumeric()
print(f"\nCODIGO_ENTIDAD - registros con valor no numerico: {mask_no_numerico.sum():,}")
if mask_no_numerico.sum() > 0:
    display(df[mask_no_numerico]['CODIGO_ENTIDAD'].value_counts().head(10).rename('frecuencia').to_frame())

# FECHA_CORTE: cuantos registros tienen cada formato
mask_iso = df['FECHA_CORTE'].str.match(r'^\d{4}-\d{2}-\d{2}$', na=False)
mask_col = df['FECHA_CORTE'].str.match(r'^\d{2}/\d{2}/\d{4}$', na=False)
mask_otro = ~mask_iso & ~mask_col
print(f"\nFECHA_CORTE - formato yyyy-mm-dd: {mask_iso.sum():,}")
print(f"FECHA_CORTE - formato dd/mm/yyyy: {mask_col.sum():,}")
print(f"FECHA_CORTE - otros / no reconocidos: {mask_otro.sum():,}")
if mask_otro.sum() > 0:
    print("Ejemplos de formatos no reconocidos:")
    print(df[mask_otro]['FECHA_CORTE'].unique()[:10])

### Conclusion 1.2



### 1.3 Cardinalidad y valores unicos en campos categoricos

Cuantifica cuantos valores unicos hay en cada campo categorico clave, detecta espacios extra (inicio, final, internos multiples) y verifica que `NOMBRE_UNIDAD_CAPTURA` solo contenga los dos valores validos del modelo de negocio.

In [ ]:
categoricos = ['TIPO_ENTIDAD', 'NOMBRE_ENTIDAD', 'NOMBRE_UNIDAD_CAPTURA', 'PRODUCTO', 'MOTIVO']

resumen_cardinalidad = pd.DataFrame({
    'campo': categoricos,
    'valores_unicos': [df[c].nunique() for c in categoricos],
    'registros_con_espacios_extra': [
        (df[c].str.strip().str.replace(r'\s+', ' ', regex=True) != df[c]).sum()
        for c in categoricos
    ]
})
display(resumen_cardinalidad)

# Verificar que NOMBRE_UNIDAD_CAPTURA solo tiene los 2 valores validos
valores_validos_unidad = {'ENTIDAD VIGILADA', 'DEFENSORES DEL CONSUMIDOR FINANCIERO'}
valores_reales_unidad = set(df['NOMBRE_UNIDAD_CAPTURA'].dropna().unique())
valores_inesperados = valores_reales_unidad - valores_validos_unidad
print(f"\nNOMBRE_UNIDAD_CAPTURA - valores esperados presentes: {valores_validos_unidad & valores_reales_unidad}")
print(f"NOMBRE_UNIDAD_CAPTURA - valores inesperados: {valores_inesperados if valores_inesperados else 'ninguno'}")

print("\nTop 10 por campo:")
for col in categoricos:
    print(f"\n  {col}:")
    display(df[col].value_counts().head(10).rename('frecuencia').to_frame())

### Conclusion 1.3



### 1.4 Duplicados por clave de negocio

El enunciado define duplicado exacto como igual combinacion de `TIPO_ENTIDAD` + `CODIGO_ENTIDAD` + `FECHA_CORTE` + `PRODUCTO` + `MOTIVO`. Se cuantifican tanto los registros involucrados como los redundantes.

In [ ]:
n_involucrados = df.duplicated(subset=COLS_CLAVE, keep=False).sum()
n_redundantes = df.duplicated(subset=COLS_CLAVE, keep='first').sum()

print(f"Registros involucrados en duplicados (clave de negocio): {n_involucrados:,}")
print(f"Registros redundantes (se eliminarian al deduplicar): {n_redundantes:,}")
print(f"Porcentaje sobre el total: {n_involucrados / len(df) * 100:.2f}%")

print("\nEjemplos de registros duplicados:")
display(
    df[df.duplicated(subset=COLS_CLAVE, keep=False)]
    .sort_values(COLS_CLAVE)
    .head(10)
)

### Conclusion 1.4



### 1.5 Los 7 problemas de calidad documentados

El pliego tecnico del engagement documenta 7 problemas de calidad conocidos en este dataset. Se localizan y cuantifican con cifra exacta, antes de aplicar ninguna correccion. Esta evidencia es la que justifica cada flag en la Seccion 2.

In [ ]:
# P1 - Fechas en formato no estandar (dd/mm/yyyy en lugar de yyyy-mm-dd)
mask_fecha_col = df['FECHA_CORTE'].str.match(r'^\d{2}/\d{2}/\d{4}$', na=False)
p1_count = mask_fecha_col.sum()
print(f"P1 - Fechas en formato dd/mm/yyyy: {p1_count:,} ({p1_count / len(df) * 100:.2f}%)")
print(f"     Ejemplos: {df.loc[mask_fecha_col, 'FECHA_CORTE'].unique()[:5]}")

# P2 - Nombres de entidad con capitalizacion inconsistente
mask_lower = df['NOMBRE_ENTIDAD'] != df['NOMBRE_ENTIDAD'].str.title()
p2_count = mask_lower.sum()
print(f"\nP2 - NOMBRE_ENTIDAD con capitalizacion incorrecta: {p2_count:,} ({p2_count / len(df) * 100:.2f}%)")
print(f"     Ejemplos: {df.loc[mask_lower, 'NOMBRE_ENTIDAD'].unique()[:5]}")

# P3 - Espacios extra en PRODUCTO y MOTIVO
for col in ['PRODUCTO', 'MOTIVO']:
    limpio = df[col].str.strip().str.replace(r'\s+', ' ', regex=True)
    p3_count = (df[col] != limpio).sum()
    print(f"\nP3 - Espacios extra en {col}: {p3_count:,} ({p3_count / len(df) * 100:.2f}%)")

In [ ]:
# P4 - FAVOR > FINALIZADAS (imposible matematicamente)
favor_total = df['QUEJAS_FAVOR_CONSUM_ACEP_ENT'] + df['QUEJAS_FAVOR_CONSUM_NOACEP_ENT']
p4_count = (favor_total > df['QUEJAS_FINALIZADAS']).sum()
print(f"P4 - FAVOR > FINALIZADAS: {p4_count:,} ({p4_count / len(df) * 100:.2f}%)")

# P5 - Duplicados por clave de negocio
p5_count = df.duplicated(subset=COLS_CLAVE, keep='first').sum()
print(f"\nP5 - Duplicados por clave de negocio (registros redundantes): {p5_count:,} ({p5_count / len(df) * 100:.2f}%)")

# P6 - Outliers en QUEJAS_EN_TRAMITE (metodo IQR)
q1_tramite = df['QUEJAS_EN_TRAMITE'].quantile(0.25)
q3_tramite = df['QUEJAS_EN_TRAMITE'].quantile(0.75)
umbral_iqr = q3_tramite + 1.5 * (q3_tramite - q1_tramite)
p6_count = (df['QUEJAS_EN_TRAMITE'] > umbral_iqr).sum()
print(f"\nP6 - Outliers QUEJAS_EN_TRAMITE (umbral IQR = {umbral_iqr:,.0f}): {p6_count:,} ({p6_count / len(df) * 100:.2f}%)")
print(f"     Valor maximo observado: {df['QUEJAS_EN_TRAMITE'].max():,}")

# P7 - CODIGO_ENTIDAD tipo mixto
p7_numerico = df['CODIGO_ENTIDAD'].str.isnumeric().sum()
p7_no_numerico = (~df['CODIGO_ENTIDAD'].str.isnumeric()).sum()
print(f"\nP7 - CODIGO_ENTIDAD: {p7_numerico:,} valores numericos puros, {p7_no_numerico:,} con caracteres mixtos")

### Conclusion 1.5



### 1.6 Validacion de identidades logicas PQRS

El flujo de PQRS define tres identidades matematicas que deben cumplirse en todo registro bien formado:

```
QUEJAS_RECIBIDAS
    QUEJAS_PENDIENTES
    QUEJAS_EN_TRAMITE
    QUEJAS_FINALIZADAS
        QUEJAS_FINALIZADAS_N1
        QUEJAS_FINALIZADAS_N2
            QUEJAS_FAVOR_CONSUM_ACEP_ENT
            QUEJAS_FAVOR_CONSUM_NOACEP_ENT
```

Una violacion es un **error de datos** (no una anomalia de negocio) y se registrara con el flag `FLAG_LOGICA_ROTA` en la Seccion 2. Los registros con violaciones no se eliminan - se preservan para auditoria.

In [ ]:
# Identidad 1: conservacion de quejas
id1 = df['QUEJAS_RECIBIDAS'] != (
    df['QUEJAS_FINALIZADAS'] + df['QUEJAS_EN_TRAMITE'] + df['QUEJAS_PENDIENTES']
)
print(f"Identidad 1 (RECIBIDAS = FINALIZADAS + EN_TRAMITE + PENDIENTES) - violaciones: {id1.sum():,} ({id1.mean() * 100:.2f}%)")

# Identidad 2: composicion de finalizadas
id2 = df['QUEJAS_FINALIZADAS'] != (
    df['QUEJAS_FINALIZADAS_N1'] + df['QUEJAS_FINALIZADAS_N2']
)
print(f"Identidad 2 (FINALIZADAS = N1 + N2) - violaciones: {id2.sum():,} ({id2.mean() * 100:.2f}%)")

# Identidad 3: fallos a favor no superan finalizadas
id3 = (
    df['QUEJAS_FAVOR_CONSUM_ACEP_ENT'] + df['QUEJAS_FAVOR_CONSUM_NOACEP_ENT']
) > df['QUEJAS_FINALIZADAS']
print(f"Identidad 3 (ACEP + NOACEP <= FINALIZADAS) - violaciones: {id3.sum():,} ({id3.mean() * 100:.2f}%)")

# Distribucion de violaciones simultaneas por registro
n_violaciones = id1.astype(int) + id2.astype(int) + id3.astype(int)
print(f"\nRegistros con al menos 1 violacion: {(n_violaciones > 0).sum():,} ({(n_violaciones > 0).mean() * 100:.2f}%)")
print("\nDistribucion de violaciones simultaneas por registro:")
display(n_violaciones.value_counts().sort_index().rename('registros').to_frame())

### Conclusion 1.6



### 1.7 Distribucion de variables numericas clave

`QUEJAS_RECIBIDAS` es el denominador de todas las tasas derivadas en la Seccion 3; si tiene ceros o distribucion extrema, las tasas produciran NaN. `QUEJAS_EN_TRAMITE` es el campo con outliers documentados. `QUEJAS_FINALIZADAS` cierra el flujo logico del pipeline PQRS.

In [ ]:
vars_num = ['QUEJAS_RECIBIDAS', 'QUEJAS_EN_TRAMITE', 'QUEJAS_FINALIZADAS']

display(
    df[vars_num].describe(percentiles=[.25, .50, .75, .90, .95, .99])
)

print("\nQUEJAS_RECIBIDAS - distribucion detallada:")
print(f"  Mediana:       {df['QUEJAS_RECIBIDAS'].median():,.0f}")
print(f"  Media:         {df['QUEJAS_RECIBIDAS'].mean():,.2f}")
print(f"  Percentil 99:  {df['QUEJAS_RECIBIDAS'].quantile(0.99):,.0f}")
print(f"  Maximo:        {df['QUEJAS_RECIBIDAS'].max():,.0f}")
print(f"  % con valor 0: {(df['QUEJAS_RECIBIDAS'] == 0).mean():.1%}")
print(f"  % con valor <= 5: {(df['QUEJAS_RECIBIDAS'] <= 5).mean():.1%}")

# Outliers IQR en QUEJAS_EN_TRAMITE
q1_t = df['QUEJAS_EN_TRAMITE'].quantile(0.25)
q3_t = df['QUEJAS_EN_TRAMITE'].quantile(0.75)
umbral_iqr_tramite = q3_t + 1.5 * (q3_t - q1_t)
outliers_tramite = df[df['QUEJAS_EN_TRAMITE'] > umbral_iqr_tramite]

print(f"\nQUEJAS_EN_TRAMITE - umbral IQR: {umbral_iqr_tramite:,.0f} | Outliers: {len(outliers_tramite):,} ({len(outliers_tramite) / len(df) * 100:.2f}%)")
print("Top 5 valores extremos:")
display(
    outliers_tramite.nlargest(5, 'QUEJAS_EN_TRAMITE')[['NOMBRE_ENTIDAD', 'FECHA_CORTE', 'QUEJAS_EN_TRAMITE']]
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

datos_pos = df.loc[df['QUEJAS_RECIBIDAS'] > 0, 'QUEJAS_RECIBIDAS']
axes[0].hist(datos_pos, bins=60, color='steelblue', edgecolor='white')
axes[0].set_xscale('log')
axes[0].axvline(datos_pos.median(), color='orange', linestyle='--', label=f'Mediana: {datos_pos.median():,.0f}')
axes[0].axvline(datos_pos.quantile(0.95), color='red', linestyle='--', label=f'P95: {datos_pos.quantile(0.95):,.0f}')
axes[0].set_title('Distribucion QUEJAS_RECIBIDAS (escala log, excluye ceros)')
axes[0].set_xlabel('Quejas recibidas')
axes[0].legend()

axes[1].boxplot(df['QUEJAS_EN_TRAMITE'], vert=False, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.6))
axes[1].axvline(umbral_iqr_tramite, color='red', linestyle='--', label=f'Umbral IQR: {umbral_iqr_tramite:,.0f}')
axes[1].set_title('QUEJAS_EN_TRAMITE - Boxplot con umbral IQR')
axes[1].set_xlabel('Quejas en tramite')
axes[1].legend()

plt.tight_layout()
plt.show()

### Conclusion 1.7



### 1.8 Preguntas de negocio exploratorias

Exploracion sin transformar datos. Los resultados aqui son sobre el dataset crudo; los valores definitivos se calcularan sobre el dataset limpio en la Seccion 6.

In [ ]:
# Pregunta 1 - Top 10 productos por volumen de quejas recibidas
total_quejas = df['QUEJAS_RECIBIDAS'].sum()
top_productos = (
    df.groupby('PRODUCTO')['QUEJAS_RECIBIDAS']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)
top_productos['pct_total'] = (top_productos['QUEJAS_RECIBIDAS'] / total_quejas * 100).round(2)
top_productos['pct_acumulado'] = top_productos['pct_total'].cumsum().round(2)
print("Pregunta 1 - Top 10 productos por QUEJAS_RECIBIDAS:")
display(top_productos)

In [ ]:
# Pregunta 2 - Tasa de resolucion a favor del consumidor por NOMBRE_UNIDAD_CAPTURA
tasa_por_canal = (
    df.groupby('NOMBRE_UNIDAD_CAPTURA')
    .agg(
        quejas_recibidas=('QUEJAS_RECIBIDAS', 'sum'),
        quejas_finalizadas=('QUEJAS_FINALIZADAS', 'sum'),
        favor_acep=('QUEJAS_FAVOR_CONSUM_ACEP_ENT', 'sum'),
        favor_noacep=('QUEJAS_FAVOR_CONSUM_NOACEP_ENT', 'sum')
    )
    .assign(
        favor_total=lambda x: x['favor_acep'] + x['favor_noacep'],
        tasa_favor=lambda x: np.where(
            x['quejas_finalizadas'] > 0,
            (x['favor_acep'] + x['favor_noacep']) / x['quejas_finalizadas'],
            np.nan
        )
    )
    .reset_index()
)
print("Pregunta 2 - Tasa de resolucion a favor del consumidor por canal:")
display(tasa_por_canal)

In [ ]:
# Pregunta 3 - Entidades con QUEJAS_EN_TRAMITE > QUEJAS_RECIBIDAS en el mismo periodo
mask_tramite_mayor = df['QUEJAS_EN_TRAMITE'] > df['QUEJAS_RECIBIDAS']
registros_afectados = mask_tramite_mayor.sum()
entidades_afectadas = df[mask_tramite_mayor]['NOMBRE_ENTIDAD'].nunique()

print(f"Pregunta 3 - Registros donde EN_TRAMITE > RECIBIDAS: {registros_afectados:,}")
print(f"Entidades distintas afectadas: {entidades_afectadas}")
print("\nDetalle por entidad (top 10 por frecuencia):")
display(
    df[mask_tramite_mayor]
    .groupby('NOMBRE_ENTIDAD')['QUEJAS_EN_TRAMITE']
    .count()
    .sort_values(ascending=False)
    .head(10)
    .rename('registros_afectados')
    .to_frame()
)

### Conclusion 1.8

